# 01 · Data acquisition

The pilot uses the two paired visible-thermal benchmarks that are freely available: **LLVIP** (15 488 aligned pairs of pedestrians, mostly at night) and **M3FD** (4 200 aligned pairs, six classes, mixed scenes). This notebook gets them onto the machine and checks they are complete. Both are licensed for non-commercial research; do not make your Kaggle copies public.

| | |
|---|---|
| **Kaggle settings** | Accelerator **GPU T4 x2**, Internet **on** |
| **Inputs to attach** | `cffm-net-pilot`, plus LLVIP and M3FD (see below) |
| **Expected runtime** | a few minutes (longer if downloading) |
| **Produces** | verified raw datasets |

## How to get the data onto Kaggle

**Option A (recommended).** Download the official archives on your computer and upload each as a **private**
Kaggle dataset (*Datasets → New Dataset → upload the zip*; Kaggle unzips it). Then attach both here
(*Add Input → Your Work*).

* LLVIP: official page <https://bupt-ai-cz.github.io/LLVIP/> (links to the GitHub release). Use the
  annotations corrected in February 2023 (the current release has them).
* M3FD: from the TarDAL repository <https://github.com/JinyuanLiu-CV/TarDAL> (the *M3FD_Detection* part:
  `Vis/`, `Ir/`, `Annotation/`).

**Option B.** Attach an existing public Kaggle copy (search for "LLVIP" / "M3FD"). Check that it has the
official folder layout and the corrected LLVIP labels.

**Option C.** Paste a public Google Drive link or file ID below and download inside this notebook.

The code finds the datasets by their folder layout, wherever they are mounted, so names do not matter.

In [ ]:
# --------------------------------------------------------------------------------------------
# Setup: the same cell opens every notebook. It finds the code, installs it, and checks the GPU.
#
# On Kaggle the uploaded cffm-net-pilot.zip appears, already unzipped, somewhere under
# /kaggle/input. Inputs are read-only, so we copy the project to /kaggle/working and install
# it from there ("pip install -e" makes `import cffm` work in this kernel, in DataLoader
# workers, and in the extra processes Ultralytics starts when it trains on two GPUs).
# Locally, notebooks live in cffm-net-pilot/notebooks/<phase>/; the project is the first folder up
# that contains src/cffm.
# --------------------------------------------------------------------------------------------
import glob, shutil, subprocess, sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    # up to five levels deep: /kaggle/input/<slug>/, /kaggle/input/<slug>/cffm-net-pilot/,
    # /kaggle/input/datasets/<owner>/<slug>/..., whichever layout this Kaggle version uses
    hits = [Path(p).parent for k in range(1, 6) for p in glob.glob("/kaggle/input" + "/*" * k + "/pyproject.toml")
            if (Path(p).parent / "src" / "cffm").is_dir()]
    assert hits, "Add the 'cffm-net-pilot' dataset (the uploaded zip) as an input to this notebook."
    PROJECT = Path("/kaggle/working/cffm-net-pilot")
    if not PROJECT.exists():
        shutil.copytree(hits[0], PROJECT, copy_function=shutil.copyfile)
        for p in [PROJECT, *PROJECT.rglob("*")]:     # the input was read-only; our copy must be writable
            p.chmod(p.stat().st_mode | 0o200)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.171",
                    "faster-coco-eval>=1.6.7", "cloudpickle"], check=True)   # cloudpickle: two-GPU launcher
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(PROJECT)], check=True)
else:
    PROJECT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "cffm").is_dir())
sys.path.insert(0, str(PROJECT / "src"))

import cffm
from cffm import env as cenv, pipeline
print("cffm", cffm.__version__, "imported from", Path(cffm.__file__).parent)   # must be inside PROJECT
env = cenv.setup()                       # prints platform, GPUs and paths; moves into the project folder
plan = pipeline.load_plan(env.project)   # configs/pilot.yaml: every run and its settings

In [ ]:
# Option C only: paste Google Drive links or file IDs of the official archives (leave empty otherwise).
LLVIP_GDRIVE = ""
M3FD_GDRIVE = ""
if LLVIP_GDRIVE:
    pipeline.download_gdrive(LLVIP_GDRIVE, env.raw / "llvip")
if M3FD_GDRIVE:
    pipeline.download_gdrive(M3FD_GDRIVE, env.raw / "m3fd")

## Where are they?

In [ ]:
found = {name: pipeline.locate_raw(name, env) for name in ("llvip", "m3fd")}
for name, p in found.items():
    print(f"{name:6s} -> {p if p else 'NOT FOUND: attach it as an input (see the options above)'}")

## Are they complete?

Counts per folder, whether every image has its partner and its annotation, and one parsed annotation to see that the class names are what the converter expects.

In [ ]:
from cffm.data import M3FD_NAMES, _parse_voc

def report_llvip(root):
    for split in ("train", "test"):
        vis = sorted((root / "visible" / split).glob("*.jpg"))
        ir = {p.name for p in (root / "infrared" / split).glob("*.jpg")}
        ann = {p.stem for p in (root / "Annotations").glob("*.xml")}
        paired = [v for v in vis if v.name in ir and v.stem in ann]
        print(f"LLVIP {split:5s}: {len(vis)} visible, {len(ir)} infrared, {len(paired)} complete pairs")
    x = next((root / "Annotations").glob("*.xml"))
    print("sample annotation", x.name, _parse_voc(x, ["person"])[:3])

def report_m3fd(root):
    from cffm.data import _find_dir
    vis, ir, ann = _find_dir(root, ["Vis", "visible"]), _find_dir(root, ["Ir", "infrared"]), \
        _find_dir(root, ["Annotation", "Annotations", "Labels"])
    nv, ni, na = (len(list(d.iterdir())) for d in (vis, ir, ann))
    print(f"M3FD: {nv} visible, {ni} infrared, {na} annotations")
    x = next(ann.glob("*.xml"))
    print("sample annotation", x.name, _parse_voc(x, M3FD_NAMES)[:3])

if found["llvip"]: report_llvip(found["llvip"])
if found["m3fd"]: report_m3fd(found["m3fd"])

## One pair from each

If the two images below do not show the same scene, the pairing is wrong and every result downstream would be meaningless.

In [ ]:
import cv2, matplotlib.pyplot as plt
from cffm.data import _find_dir

def show_raw_pair(vis_path, ir_path, title):
    v, i = cv2.imread(str(vis_path))[..., ::-1], cv2.imread(str(ir_path), cv2.IMREAD_GRAYSCALE)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    ax[0].imshow(v); ax[0].set_title(f"{title}: visible"); ax[1].imshow(i, cmap="gray"); ax[1].set_title("thermal")
    for a in ax: a.axis("off")
    plt.show()

if found["llvip"]:
    v = sorted((found["llvip"] / "visible" / "test").glob("*.jpg"))[100]
    show_raw_pair(v, found["llvip"] / "infrared" / "test" / v.name, "LLVIP")
if found["m3fd"]:
    vd, idr = _find_dir(found["m3fd"], ["Vis", "visible"]), _find_dir(found["m3fd"], ["Ir", "infrared"])
    v = sorted(vd.iterdir())[50]
    show_raw_pair(v, next(idr.glob(v.stem + ".*")), "M3FD")